# Ingeniería de features — construir y verificar variables nuevas

## Qué es esto, y en qué se distingue de la EDA

`04_eda_targets.ipynb` a `09_eda_jugador_destacado.ipynb` dejaron un inventario real de qué
variables existen y cómo se comportan. Este notebook no repite ese inventario — **construye**
variables nuevas a partir de él: transformaciones, ratios de eficiencia, agregados de equipo, un
indicador de cambio de quarterback titular, y combinaciones entre variables que tengan una razón
concreta para probarse juntas. Cada una se verifica fila por fila sobre un caso real, con el mismo
criterio que ya se usó para probar que `agregar_promedios_jugador()` no tiene fuga
(`03_features.ipynb`).

La pregunta que responde: **¿estas funciones nuevas calculan lo que dicen calcular, sin fuga de
datos?** Cuáles de estas variables (más las que ya existían) realmente ayudan a predecir se
responde después, en `11_seleccion_features.ipynb` — aquí solo se construye y se verifica.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, features_exploratorio

seasons = list(range(2016, 2026))
stats = data.cargar_stats_semanales(seasons)
jugadores = data.cargar_jugadores()
wr = stats[stats["position"] == "WR"].copy()
print("temporadas:", seasons[0], "-", seasons[-1])
print("filas WR:", wr.shape[0])

temporadas: 2016 - 2025
filas WR: 23610


## 1. Ratios de eficiencia — un ejemplo real antes que la fórmula

Hasta ahora, `features.py` solo mide **volumen** (cuántas yardas, cuántas recepciones). Le falta
la otra mitad de la historia: **eficiencia** — de lo que le llegó, cuánto convirtió.

Un caso real, la semana 1 de 2025 de Ja'Marr Chase: le llegaron **5 targets**, atrapó **2**, para
**26 yardas** de recepción, sobre pases que viajaron **88 yardas de aire** en total antes de
llegar a algún receptor suyo. De ahí salen 3 preguntas distintas, todas útiles:

- **¿Qué tan seguido atrapa lo que le tiran?** `catch_rate` = 2 / 5 = **0.40** (40% esa semana).
- **¿Cuántas yardas produce por cada oportunidad?** `yards_per_target` = 26 / 5 = **5.2** yardas
  por target.
- **¿Qué tan profundos son los pases que le tiran, en promedio?** `air_yards_per_target` =
  88 / 5 = **17.6** yardas de aire por target.

Ninguna de las 3 existe hoy en `features.py` — solo se tienen los volúmenes crudos.

In [2]:
wr = features.calcular_ratios_eficiencia(wr)
chase = wr[(wr["player_display_name"] == "Ja'Marr Chase") & (wr["season"] == 2025) & (wr["week"] == 1)]
chase[["player_display_name", "targets", "receptions", "receiving_yards", "receiving_air_yards",
       "catch_rate", "yards_per_target", "air_yards_per_target"]]

,player_display_name,targets,receptions,receiving_yards,receiving_air_yards,catch_rate,yards_per_target,air_yards_per_target
156313,Ja'Marr Chase,5,2,26,88,0.4,5.2,17.6


Los 3 números coinciden exactamente con la cuenta de arriba. Falta un caso al margen: **¿qué pasa
un juego en el que no le tiran nada?** Dividir entre 0 no debe ni tronar ni inventar un 0 falso —
un jugador sin targets no tiene una tasa de acierto que calcular, el valor correcto es "no
aplica", no cero.

In [3]:
sin_targets = wr[wr["targets"] == 0]
print("filas con targets=0:", sin_targets.shape[0])
print("catch_rate nulo en todas:", sin_targets["catch_rate"].isna().all())
print("yards_per_target nulo en todas:", sin_targets["yards_per_target"].isna().all())

filas con targets=0: 3110
catch_rate nulo en todas: True
yards_per_target nulo en todas: True


Confirmado: las 3 quedan `NaN`, nunca `0` ni un error — se preservan como "no aplica", que es lo
correcto para promediarlas después sin distorsionar el promedio de un jugador con ceros falsos.

## 2. `racr` acotado — la variable inestable de Fase 3, con un tope real

`05_eda_perfil_variables.ipynb` ya mostró que `racr` (yardas de recepción / yardas de aire) es
numéricamente inestable: cuando el denominador se acerca a cero, el cociente se dispara. El caso
más extremo encontrado ahí fue real, no un caso hipotético: **Khalil Shakir, 2025 semana 9** —
43 yardas de recepción sobre apenas 1 yarda de aire, `racr = 43.0`.

En vez de descartar `racr` por completo, se prueba una versión **acotada** (winsorizada al
percentil 99 real de la columna, no a un número inventado) — si esa versión sí aporta algo en la
evaluación conjunta (`11_seleccion_features.ipynb`), se queda; si no, se descarta con la misma
evidencia de siempre.

In [4]:
wr["racr"] = (wr["receiving_yards"] / wr["receiving_air_yards"]).where(wr["receiving_air_yards"] > 0)
shakir = wr[(wr["player_display_name"] == "Khalil Shakir") & (wr["season"] == 2025) & (wr["week"] == 9)]
print(shakir[["player_display_name", "receiving_yards", "receiving_air_yards", "racr"]].to_string(index=False))

wr = features_exploratorio.acotar_variable(wr, "racr", percentil=0.99)
tope = wr["racr"].quantile(0.99)
print(f"\npercentil 99 real de racr: {tope:.3f}")
shakir2 = wr[(wr["player_display_name"] == "Khalil Shakir") & (wr["season"] == 2025) & (wr["week"] == 9)]
print(shakir2[["racr", "racr_acotado"]].to_string(index=False))

player_display_name  receiving_yards  receiving_air_yards  racr
      Khalil Shakir               43                    1  43.0

percentil 99 real de racr: 5.667
 racr  racr_acotado
 43.0      5.666667


El valor crudo (43.0) queda topado en {:.3f} — deja de ser un valor extremo aislado capaz de
arrastrar el promedio de todo un jugador, sin descartar la fila por completo.

## 3. Volatilidad reciente — no solo el promedio, también la consistencia

`09_eda_jugador_destacado.ipynb` mostró que la semana boom de Ja'Marr Chase fue volumen +
profundidad + acierto juntos — un jugador puede tener el mismo promedio que otro y aun así ser
mucho más impredecible semana a semana. Un promedio no distingue "siempre produce lo mismo" de
"una semana explota y la siguiente desaparece" — hace falta medir también la **dispersión**
reciente, no solo el nivel.

Se agrega la desviación estándar de las últimas 5 apariciones (rezagada, mismo patrón que
`agregar_promedios_jugador`) — un número más alto significa más boom-bust, uno más bajo significa
más consistencia semana a semana.

In [5]:
wr = features_exploratorio.agregar_volatilidad_jugador(wr, ["receiving_yards"], ventana=5)
chase = wr[(wr["player_display_name"] == "Ja'Marr Chase") & (wr["season"] == 2025)].sort_values("week")
chase[["week", "receiving_yards", "receiving_yards_volatilidad5"]].head(8)

,week,receiving_yards,receiving_yards_volatilidad5
156313,1,26,35.786869
157388,2,165,32.000000
158485,3,50,49.220931
159593,4,23,53.564914
160637,5,110,59.636398
161620,6,94,61.349002
162644,7,161,55.039077
163614,8,91,53.659109


La primera fila de 2025 ya trae un número (no `NaN`) — porque la ventana de 5 apariciones no se
reinicia por temporada, igual que `last5_avg`, así que arrastra las últimas semanas de 2024. Es la
misma convención ya usada y documentada para las ventanas de forma reciente, no un caso especial
nuevo.

## 4. Edad y experiencia — la transformación no lineal que faltaba

`06_eda_general.ipynb` encontró que la correlación lineal de años de experiencia contra los 3
targets es prácticamente cero, pero la relación real tiene forma de curva: sube hasta 3-5 años de
experiencia y baja después. Una correlación lineal no detecta eso — pero un término cuadrático o
una variable de bucket sí pueden, si se le da la oportunidad a un modelo de usarlos.

Se agregan aquí: `edad`/`anios_experiencia` (hoy no integradas a ningún feature del jugador, solo
viven sueltas en `cargar_jugadores()`), más el término cuadrático y el bucket — reusando
exactamente los mismos cortes (`Rookie`/`1-2 años`/`3-5 años`/`6+ años`) ya validados con datos
reales en ese notebook, no inventados de nuevo.

In [6]:
wr = features.agregar_edad_experiencia(wr, jugadores)
wr = features.agregar_experiencia_no_lineal(wr)

carrera_larga = wr[wr["anios_experiencia"] >= 10].drop_duplicates("player_display_name")
carrera_larga = carrera_larga.sort_values("anios_experiencia", ascending=False)
carrera_larga[["player_display_name", "anios_experiencia", "anios_experiencia_sq",
               "anios_experiencia_bucket", "edad"]].head(5)

,player_display_name,anios_experiencia,anios_experiencia_sq,anios_experiencia_bucket,edad
0,Steve Smith,15,225,6+ años,37.308693
20,Anquan Boldin,13,169,6+ años,35.912389
14,Andre Johnson,13,169,6+ años,35.143053
36,Larry Fitzgerald,12,144,6+ años,33.004791
113,Vincent Jackson,11,121,6+ años,33.631759


Jugadores de carrera larga y reconocida (Larry Fitzgerald, Steve Smith) caen correctamente en el
bucket `6+ años`, con el cuadrado de sus años de experiencia calculado sobre el valor real, no
sobre un promedio.

## 5. `draft_pick` — la variable nunca antes usada, con su NaN estructural resuelto

`08_eda_multivariable.ipynb` encontró que `draft_pick` correlaciona más que edad o experiencia
(-0.22 recepciones, -0.21 yardas) — nunca se había usado en el proyecto. El obstáculo es que es
`NaN` para **54% de los WR históricos** (`docs/data/DATASHEET.md`), y ese `NaN` no es un dato
faltante por error — es la ausencia real de un pick, porque el jugador no fue drafteado.

Imputar con la media o la mediana trataría a un no drafteado como a un jugador "promedio", lo
cual no es cierto — por definición, no ser drafteado es peor que el último pick real. Se imputa
como **el peor pick real observado + 1**, calculado sobre el dato mismo (no un número fijo
escrito a mano).

In [7]:
wr = features.agregar_draft_pick(wr, jugadores)
peor_que_ultimo = jugadores["draft_pick"].max() + 1
print(f"peor pick real + 1 = {peor_que_ultimo}")

chase = wr[(wr["player_display_name"] == "Ja'Marr Chase")].iloc[0]
print(f"\nJa'Marr Chase (sí drafteado, pick real conocido: 5): draft_pick = {chase['draft_pick']}")

no_drafteados = wr[wr["draft_pick"] == peor_que_ultimo]
ejemplo = no_drafteados.iloc[0]
print(f"{ejemplo['player_display_name']} (no drafteado, {no_drafteados['player_id'].nunique()} "
      f"jugadores distintos en este caso): draft_pick imputado = {ejemplo['draft_pick']}")

peor pick real + 1 = 473.0

Ja'Marr Chase (sí drafteado, pick real conocido: 5): draft_pick = 5.0
Eric Weems (no drafteado, 282 jugadores distintos en este caso): draft_pick imputado = 473.0


El jugador drafteado conserva su pick real; al no drafteado no se le inventa un pick "bueno" —
queda estrictamente peor que cualquier pick real de la historia de los datos.

## 6. Cambio de quarterback titular — investigar antes de construir

`08_eda_multivariable.ipynb` encontró que un cambio de QB titular golpea la producción del
receptor (~16% menos yardas, 594 casos) más que la calidad continua del QB. Ahí mismo hay un
prototipo ad-hoc: el pasador con más intentos esa semana por equipo. Antes de convertirlo en una
función de verdad, hace falta revisar 3 casos límite con datos reales — no asumir que el
prototipo ya los cubre bien.

**a) ¿Qué tan seguido empatan dos pasadores del mismo equipo en la misma semana?**

In [8]:
qb = stats[(stats["position"] == "QB") & (stats["attempts"] > 0)][
    ["season", "week", "team", "player_display_name", "attempts", "passing_epa"]
].copy()
grp = qb.groupby(["season", "week", "team"])
maximo = grp["attempts"].transform("max")
empatados_en_max = grp["attempts"].transform(lambda s: (s == s.max()).sum())
empates = qb[(qb["attempts"] == maximo) & (empatados_en_max > 1)]
print(f"equipo-semana con empate en el máximo de attempts: {empates.shape[0] // 2} de {grp.ngroups} "
      f"({100 * (empates.shape[0] // 2) / grp.ngroups:.2f}%)")
empates.sort_values(["season", "week", "team"]).head(6)

equipo-semana con empate en el máximo de attempts: 7 de 5269 (0.13%)


,season,week,team,player_display_name,attempts,passing_epa
19952,2017,4,TEN,Matt Cassel,10,-22.404021
20666,2017,4,TEN,Marcus Mariota,10,-1.115625
105380,2022,4,PIT,Mitchell Trubisky,13,-6.632850
106078,2022,4,PIT,Kenny Pickett,13,1.425465
126824,2023,8,NYG,Tyrod Taylor,7,-10.349501
127736,2023,8,NYG,Tommy DeVito,7,-6.947000


Rarísimo (0.13% de los equipo-semana) pero real — no se puede ignorar en silencio. Se desempata
por `passing_epa` (quién jugó mejor esa semana), no por el orden en que vengan las filas.

**b) ¿Existen cambios de titular reales, no solo teóricos?**

In [9]:
qb_sorted = qb.sort_values(["attempts", "passing_epa"], ascending=False)
qb_starter = qb_sorted.drop_duplicates(subset=["season", "week", "team"]).sort_values(["team", "season", "week"])
qb_starter["titular_anterior"] = qb_starter.groupby("team")["player_display_name"].shift(1)
qb_starter["semana_anterior"] = qb_starter.groupby("team")["week"].shift(1)
qb_starter["temporada_anterior"] = qb_starter.groupby("team")["season"].shift(1)
cambios = qb_starter[
    (qb_starter["player_display_name"] != qb_starter["titular_anterior"])
    & qb_starter["titular_anterior"].notna()
    & (qb_starter["season"] == qb_starter["temporada_anterior"])
]
print(f"cambios de titular reales (misma temporada): {cambios.shape[0]}")

cambios de titular reales (misma temporada): 532


532 casos reales en 10 temporadas — confirma que vale la pena construir el feature, no es un
evento anecdótico.

**c) ¿Un traspaso de equipo a media temporada se confunde con un cambio de QB?**

In [10]:
wr_sorted = wr.sort_values(["player_id", "season", "week"])
wr_sorted["team_anterior"] = wr_sorted.groupby("player_id")["team"].shift(1)
wr_sorted["season_anterior"] = wr_sorted.groupby("player_id")["season"].shift(1)
traspasos = wr_sorted[
    (wr_sorted["team"] != wr_sorted["team_anterior"])
    & wr_sorted["team_anterior"].notna()
    & (wr_sorted["season"] == wr_sorted["season_anterior"])
]
print(f"filas de WR con cambio de equipo dentro de la misma temporada: {traspasos.shape[0]}")

filas de WR con cambio de equipo dentro de la misma temporada: 105


105 casos reales — sí ocurre, y con la lógica de "compara contra tu aparición anterior" un
traspaso de equipo también se leería como "cambió el QB titular", porque casi siempre es cierto a
la vez. En vez de ignorarlo, se construyen **dos banderas separadas** — `cambio_qb_titular` y
`cambio_equipo` — para no mezclar dos señales distintas en una sola variable.

### Construcción, verificada sobre 2 casos reales

**Cleveland, 2024**: carrusel de quarterbacks ya documentado — Deshaun Watson (semanas 1-6),
Dorian Thompson-Robinson (semana 7), Jameis Winston (semanas 8-15). El cambio debe marcarse en las
semanas 7 y 8 para cada receptor del equipo, y en ninguna otra.

In [11]:
qb_titular = data.identificar_qb_titular(stats)
wr = features_exploratorio.agregar_cambio_qb_titular(wr, qb_titular)

cle = wr[(wr["season"] == 2024) & (wr["team"] == "CLE") & (wr["week"].between(6, 9))]
cle.sort_values(["player_display_name", "week"])[
    ["player_display_name", "week", "qb_nombre", "cambio_qb_titular", "cambio_equipo"]
]

,player_display_name,week,qb_nombre,cambio_qb_titular,cambio_equipo
5871,Amari Cooper,6,Deshaun Watson,False,False
22126,Cedric Tillman,6,Deshaun Watson,False,False
22127,Cedric Tillman,7,Dorian Thompson-Robinson,True,False
22128,Cedric Tillman,8,Jameis Winston,True,False
22129,Cedric Tillman,9,Jameis Winston,False,False
19660,Elijah Moore,6,Deshaun Watson,False,False
19661,Elijah Moore,7,Dorian Thompson-Robinson,True,False
19662,Elijah Moore,8,Jameis Winston,True,False
19663,Elijah Moore,9,Jameis Winston,False,False
19224,Jaelon Darden,6,Deshaun Watson,False,False


Exactamente lo esperado, para los 4 receptores del equipo por igual: `cambio_qb_titular = True`
en semana 7 (Watson → Thompson-Robinson) y en semana 8 (→ Winston), `False` el resto — y
`cambio_equipo = False` en todas, porque ningún jugador cambió de equipo, solo cambió el QB. Esa
separación es justo el punto.

**Ke'Shawn Williams, 2025**: traspaso real de equipo (PIT → CIN) entre la semana 13 y la 16 (bye
de por medio) — aquí `cambio_equipo` debe encenderse, y con él, casi seguro, también
`cambio_qb_titular`.

In [12]:
williams = wr[wr["player_display_name"] == "Ke'Shawn Williams"].sort_values(["season", "week"])
williams[williams["season"] == 2025][["week", "team", "qb_nombre", "cambio_qb_titular", "cambio_equipo"]]

,week,team,qb_nombre,cambio_qb_titular,cambio_equipo
23441,6,PIT,Aaron Rodgers,NaN,NaN
23442,7,PIT,Aaron Rodgers,False,False
23443,8,PIT,Aaron Rodgers,False,False
23444,9,PIT,Aaron Rodgers,False,False
23445,10,PIT,Aaron Rodgers,False,False
23446,11,PIT,Mason Rudolph,True,False
23447,12,PIT,Mason Rudolph,False,False
23448,13,PIT,Aaron Rodgers,True,False
23449,16,CIN,Joe Burrow,True,True
23450,17,CIN,Joe Burrow,False,False


Confirmado: en la semana 16, `cambio_equipo = True` y `cambio_qb_titular = True` a la vez — las
dos banderas capturan lo que de verdad pasó, sin forzar una sola variable a explicar dos eventos
distintos.

## 7. Ofensiva de equipo — verificación de no fuga, con aritmética exacta

`07_eda_equipos.ipynb` encontró 0.40 de estabilidad año-a-año en la producción ofensiva de un
equipo — pero esa cifra es **anual**. La versión que de verdad se puede usar como feature semanal
(rezagada, sin ver el futuro) ya se probó en `08_eda_multivariable.ipynb` y dio apenas ~0.08 de
correlación individual — la expectativa correcta para lo que sigue es esa, no el 0.40.

La construcción reutiliza `agregar_promedios_jugador()` a nivel equipo — no duplica la lógica de
`shift`/`rolling` ya verificada. La prueba de no-fuga es aritmética directa: sumar a mano las
yardas de recepción de todo un equipo en sus primeras semanas, dividir entre el número de
semanas, y comparar contra lo que produce la función.

In [13]:
wr = features_exploratorio.agregar_ofensiva_equipo(wr, columnas=("receiving_yards",), ventanas=(3, 5))

cin_1a4 = wr[(wr["team"] == "CIN") & (wr["season"] == 2025) & (wr["week"] < 5)]
suma_manual = cin_1a4.drop_duplicates(["week"])  # evitar doble conteo si un jugador aparece 2 veces
suma_manual = cin_1a4.groupby("week")["receiving_yards"].sum()
promedio_manual = suma_manual.sum() / suma_manual.shape[0]
print(f"suma manual de receiving_yards de CIN, semanas 1-4: {suma_manual.sum()}, "
      f"entre {suma_manual.shape[0]} semanas = {promedio_manual:.2f}")

semana_5 = wr[(wr["team"] == "CIN") & (wr["season"] == 2025) & (wr["week"] == 5)]
print(f"equipo_receiving_yards_season_avg en semana 5 (no debe ver la semana 5 misma): "
      f"{semana_5['equipo_receiving_yards_season_avg'].iloc[0]:.2f}")

suma manual de receiving_yards de CIN, semanas 1-4: 456, entre 4 semanas = 114.00
equipo_receiving_yards_season_avg en semana 5 (no debe ver la semana 5 misma): 114.00


Coinciden exactamente — el promedio que ve un jugador en la semana 5 es aritméticamente el
promedio de las semanas 1-4 de su equipo, sin que la producción de la semana 5 (la que se quiere
predecir) se cuele en su propio feature.

## 8. Combinaciones — solo las que tienen una razón concreta

No combinaciones arbitrarias: cada una responde una pregunta específica que no se puede contestar
mirando las variables por separado.

In [14]:
wr = features_exploratorio.agregar_interaccion(
    wr, "target_share", "equipo_receiving_yards_season_avg", nombre="target_share_x_ofensiva_equipo"
)
wr = features_exploratorio.agregar_interaccion(
    wr, "draft_pick", "anios_experiencia", nombre="draft_pick_x_experiencia"
)
wr["cambio_qb_titular_num"] = wr["cambio_qb_titular"].astype("float")
wr = features_exploratorio.agregar_interaccion(
    wr, "cambio_qb_titular_num", "target_share", nombre="cambio_qb_x_target_share"
)

cols_interaccion = ["target_share_x_ofensiva_equipo", "draft_pick_x_experiencia", "cambio_qb_x_target_share"]
wr[cols_interaccion].describe()

,target_share_x_ofensiva_equipo,draft_pick_x_experiencia,cambio_qb_x_target_share
count,22183.000000,23610.000000,22851.000000
mean,20.946759,601.422787,0.018711
std,18.195130,862.296209,0.059859
min,0.000000,0.000000,0.000000
25%,6.184811,48.000000,0.000000
50%,16.888889,252.000000,0.000000
75%,31.521739,765.000000,0.000000
max,131.272727,6149.000000,0.600000


- **`target_share_x_ofensiva_equipo`**: ¿importa más un mismo % de participación en una ofensiva
  fuerte que en una débil? Se prueba en `11_seleccion_features.ipynb` si esta combinación explica
  algo que ninguna de las 2 variables por separado ya explicaba.
- **`draft_pick_x_experiencia`**: ¿el origen de draft pesa igual en un rookie que en un jugador ya
  establecido?
- **`cambio_qb_x_target_share`**: ¿un cambio de QB golpea más a los receptores de mayor volumen?

## Cierre

Funciones nuevas construidas y verificadas fila por fila sobre casos reales, todas en
`weekly/wr/src/features.py` (y `identificar_qb_titular` en `data.py`):

| Función | Verificada con |
|---|---|
| `calcular_ratios_eficiencia` | Ja'Marr Chase, 2025 semana 1; nulos correctos en `targets=0` |
| `acotar_variable` | Khalil Shakir, 2025 semana 9 (caso ancla de `racr` inestable) |
| `agregar_volatilidad_jugador` | Ja'Marr Chase, 2025 |
| `agregar_edad_experiencia` / `agregar_experiencia_no_lineal` | jugadores de carrera larga (Fitzgerald, Smith) |
| `agregar_draft_pick` | Ja'Marr Chase (drafteado) vs. un WR no drafteado real |
| `identificar_qb_titular` / `agregar_cambio_qb_titular` | Cleveland 2024 (cambio de QB) + Ke'Shawn Williams 2025 (traspaso de equipo) |
| `agregar_ofensiva_equipo` | Cincinnati 2025, semanas 1-4 vs. 5 (aritmética exacta) |
| `agregar_interaccion` | 3 combinaciones con razón concreta |

Lo que falta — y es la pregunta real de esta fase — es **cuáles de todas estas variables (las
nuevas y las que ya existían) de verdad ayudan a predecir cada uno de los 3 targets**. Eso se
responde en [`11_seleccion_features.ipynb`](11_seleccion_features.ipynb), no aquí.

Notebook anterior: [`09_eda_jugador_destacado.ipynb`](09_eda_jugador_destacado.ipynb) · Siguiente: [`11_seleccion_features.ipynb`](11_seleccion_features.ipynb).